# NumPy Broadcasting

Broadcasting is NumPy's way of performing operations on arrays of different shapes. It's what allows you to subtract the mean from every column in one line, or compute distance matrices without loops. Understanding broadcasting is essential for writing efficient numerical code.

## Learning Objectives

- Understand the three rules of broadcasting
- Visualize how arrays are stretched to match shapes
- Apply broadcasting to common data science tasks (centering, normalizing, distances)
- Debug broadcasting errors by understanding why they occur

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## The Problem Broadcasting Solves

Without broadcasting, you'd need explicit loops to perform operations on arrays of different shapes.

In [ ]:
# Without broadcasting: add 10 to every element
arr = np.array([1, 2, 3, 4, 5])

# The slow way (don't do this!)
result_slow = np.zeros_like(arr)
for i in range(len(arr)):
    result_slow[i] = arr[i] + 10

# With broadcasting (the NumPy way)
result_fast = arr + 10

print(f"Original: {arr}")
print(f"After adding 10: {result_fast}")

In [ ]:
# Scalar broadcasting is intuitive
arr = np.array([1, 2, 3])
print(f"arr + 10 = {arr + 10}")
print(f"arr * 2 = {arr * 2}")
print(f"arr ** 2 = {arr ** 2}")

## The Three Rules of Broadcasting

When operating on two arrays, NumPy compares their shapes element-wise, starting from the **trailing dimensions**:

1. **If arrays have different number of dimensions, the shape of the smaller one is padded with 1s on the left**
2. **Dimensions are compatible if they are equal OR one of them is 1**
3. **Arrays with size 1 along a dimension are stretched to match the other array's size**

If these conditions aren't met, you get a broadcasting error.

In [ ]:
# Rule 1: Padding with 1s on the left
a = np.ones((3, 4))        # Shape: (3, 4)
b = np.array([1, 2, 3, 4]) # Shape: (4,) -> becomes (1, 4) for comparison

print(f"a shape: {a.shape}")
print(f"b shape: {b.shape}")

result = a + b
print(f"Result shape: {result.shape}")
print(f"Result:\n{result}")

In [ ]:
# Rule 2 & 3: Stretching size-1 dimensions
a = np.array([[1], [2], [3]])  # Shape: (3, 1)
b = np.array([10, 20, 30, 40]) # Shape: (4,) -> (1, 4)

print(f"a shape: {a.shape}")
print(f"a:\n{a}")
print(f"\nb shape: {b.shape}")
print(f"b: {b}")

# Broadcasting: (3, 1) and (1, 4) -> (3, 4)
result = a + b
print(f"\nResult shape: {result.shape}")
print(f"Result:\n{result}")

## Visualizing Broadcasting

Let's visualize how a (3,1) and a (1,4) array broadcast to (3,4).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Array a: (3, 1)
a = np.array([[1], [2], [3]])
ax = axes[0]
im = ax.imshow(a, cmap='Blues', aspect='equal')
ax.set_title(f'Array A\nShape: {a.shape}')
for i in range(3):
    ax.text(0, i, a[i, 0], ha='center', va='center', fontsize=14)
ax.set_xticks([0])
ax.set_yticks([0, 1, 2])

# Array b: (1, 4)
b = np.array([[10, 20, 30, 40]])
ax = axes[1]
im = ax.imshow(b, cmap='Oranges', aspect='equal')
ax.set_title(f'Array B\nShape: {b.shape}')
for j in range(4):
    ax.text(j, 0, b[0, j], ha='center', va='center', fontsize=14)
ax.set_xticks([0, 1, 2, 3])
ax.set_yticks([0])

# Broadcast visualization (stretched)
a_broadcast = np.broadcast_to(a, (3, 4))
b_broadcast = np.broadcast_to(b, (3, 4))

ax = axes[2]
ax.imshow(a_broadcast, cmap='Blues', alpha=0.5, aspect='equal')
ax.imshow(b_broadcast, cmap='Oranges', alpha=0.5, aspect='equal')
ax.set_title(f'Broadcast\n(3,1) + (1,4) → (3,4)')
for i in range(3):
    for j in range(4):
        ax.text(j, i, f'{a[i,0]}+{b[0,j]}', ha='center', va='center', fontsize=9)
ax.set_xticks([0, 1, 2, 3])
ax.set_yticks([0, 1, 2])

# Result
result = a + b
ax = axes[3]
im = ax.imshow(result, cmap='Greens', aspect='equal')
ax.set_title(f'Result\nShape: {result.shape}')
for i in range(3):
    for j in range(4):
        ax.text(j, i, result[i, j], ha='center', va='center', fontsize=14)
ax.set_xticks([0, 1, 2, 3])
ax.set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()

## Common Broadcasting Patterns in Data Science

In [ ]:
# Pattern 1: Subtracting the mean (centering data)
# Common in preprocessing for ML

# Data matrix: 4 samples, 3 features
data = np.array([
    [10, 100, 1000],
    [20, 200, 2000],
    [30, 300, 3000],
    [40, 400, 4000]
], dtype=float)

print(f"Data shape: {data.shape}")
print(f"Data:\n{data}")

# Mean of each column (feature)
col_means = data.mean(axis=0)
print(f"\nColumn means: {col_means}")  # Shape: (3,)

# Center the data (subtract mean from each column)
# data: (4, 3), col_means: (3,) -> broadcasts to (4, 3)
centered = data - col_means
print(f"\nCentered data:\n{centered}")
print(f"Column means of centered: {centered.mean(axis=0)}")  # Should be ~0

In [ ]:
# Pattern 2: Normalizing (standardizing) data
# z = (x - mean) / std

col_stds = data.std(axis=0)
standardized = (data - col_means) / col_stds

print(f"Standardized data:\n{standardized}")
print(f"\nColumn means: {standardized.mean(axis=0)}")
print(f"Column stds: {standardized.std(axis=0)}")

In [ ]:
# Pattern 3: Row-wise operations with column vector
# E.g., multiply each row by different weights

data = np.array([[1, 2, 3], [4, 5, 6]])
row_weights = np.array([[2], [0.5]])  # Shape: (2, 1)

print(f"Data: {data.shape}")
print(f"Weights: {row_weights.shape}")

weighted = data * row_weights
print(f"Weighted:\n{weighted}")

In [ ]:
# Pattern 4: Computing pairwise differences (for distance matrices)
# This is the key insight behind efficient distance calculations

points = np.array([[0, 0], [1, 1], [2, 0]])  # 3 points in 2D
print(f"Points:\n{points}")

# To get all pairwise differences:
# Reshape points to (3, 1, 2) and (1, 3, 2)
# Then subtract to get (3, 3, 2) array of differences

diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
print(f"\nDiff shape: {diff.shape}")  # (3, 3, 2)

# Compute Euclidean distances
distances = np.sqrt((diff ** 2).sum(axis=2))
print(f"\nDistance matrix:\n{distances}")

## When Broadcasting Fails

In [ ]:
# Broadcasting fails when dimensions are incompatible
a = np.ones((3, 4))
b = np.ones((3,))  # This is (3,), not (3, 1) or (1, 3)

print(f"a shape: {a.shape}")
print(f"b shape: {b.shape}")

# This will fail:
# Comparing from right: 4 vs 3 -> NOT compatible!
try:
    result = a + b
except ValueError as e:
    print(f"\nError: {e}")

In [ ]:
# Fix: Reshape b to be broadcastable
a = np.ones((3, 4))
b = np.array([1, 2, 3])

# Option 1: Make b a column vector (3, 1)
b_col = b[:, np.newaxis]  # or b.reshape(-1, 1)
print(f"b as column: {b_col.shape}")
result = a + b_col
print(f"Result:\n{result}")

> **Gotcha: Shape (n,) vs (n, 1) vs (1, n)**  
> A 1D array of shape `(n,)` is NOT the same as a 2D array of shape `(n, 1)` or `(1, n)`.  
> Use `arr[:, np.newaxis]` or `arr.reshape(-1, 1)` to add dimensions.

In [ ]:
# Check broadcast compatibility before operating
a = np.ones((3, 4))
b = np.ones((4,))
c = np.ones((3,))

# np.broadcast_shapes tells you the result shape (or raises error)
print(f"a + b would have shape: {np.broadcast_shapes(a.shape, b.shape)}")

try:
    print(f"a + c would have shape: {np.broadcast_shapes(a.shape, c.shape)}")
except ValueError as e:
    print(f"a + c error: {e}")

## Exercises

### Exercise 1: Center and Scale

Given a data matrix, center each column (subtract column mean) and then scale so each column has unit variance. Use broadcasting—no loops!

In [ ]:
np.random.seed(42)
data = np.random.randn(100, 5) * [1, 10, 100, 1000, 10000] + [0, 50, 500, 5000, 50000]
print(f"Data shape: {data.shape}")
print(f"Column means: {data.mean(axis=0)}")
print(f"Column stds: {data.std(axis=0)}")

# Standardize the data (mean=0, std=1 for each column)
# YOUR CODE HERE

### Exercise 2: Outer Product

Given two 1D arrays `a` and `b`, compute the outer product (a matrix where element [i,j] = a[i] * b[j]) using broadcasting, not `np.outer()`.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])

# Expected result: 3x4 matrix where result[i,j] = a[i] * b[j]
# [[10, 20, 30, 40],
#  [20, 40, 60, 80],
#  [30, 60, 90, 120]]

# YOUR CODE HERE

### Exercise 3: Distance Matrix

Given two sets of points, compute the pairwise Euclidean distance matrix using broadcasting.

In [ ]:
# Set A: 3 points in 2D
points_a = np.array([[0, 0], [1, 0], [0, 1]])

# Set B: 4 points in 2D
points_b = np.array([[0, 0], [1, 1], [2, 2], [3, 3]])

# Compute 3x4 distance matrix where dist[i,j] = distance(points_a[i], points_b[j])
# YOUR CODE HERE

### Exercise 4: Debug Broadcasting Error

The following code has a broadcasting error. Fix it.

In [ ]:
# We want to add different biases to each row
data = np.random.randn(5, 3)  # 5 samples, 3 features
row_biases = np.array([0, 1, 2, 3, 4])  # 5 biases (one per row)

# This fails:
# result = data + row_biases

# Fix it to add bias[0] to row 0, bias[1] to row 1, etc.
# YOUR CODE HERE